In [1]:
# execute as if in root folder for utils and filepaths
%cd ..

c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc


C:\Users\sonja\AppData\Roaming\Python\Python311\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import random
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import json
from torch.utils.tensorboard import SummaryWriter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, auc, precision_recall_curve , average_precision_score


from utils import get_support_query_loaders, log_results_nn, NeuralNetwork, save_best_model, classification_loss, propagation_loss, getPrAucIndividualClassFinetuning

In [3]:
data = np.load('./data/data_scaled.npz')
X_finetuning = data['X_finetuning']
y_ft1 = data['y_finetuning1']
y_ft2 = data['y_finetuning2']

In [4]:
seeds = [8479, 227, 5413, 8179, 7528]

In [5]:
def label_propagation(X, Y, alpha):
    """
    X: input features
    Y: one hot vector of n x c dimensions; c is num of different labels
    alpha: hyperparameter that controls how much influence neighbors get, [0,1]: 0: predict only 0s, 1: predict only 1s
    """
    device = X.device
    # affinity matrix W
    sq = torch.sum(X ** 2, dim=1, keepdim=True)  # shape: (num_samples, 1)
    distances_squared = sq + sq.t() - 2 * (X @ X.t())
    sigma2 = torch.var(distances_squared)
    W = torch.exp(-distances_squared / (2*sigma2) )
    W.fill_diagonal_(0)

    # diagonal matrix
    row_sums = torch.sum(W, dim=1)
    D12 = torch.diag(1 / torch.sqrt(row_sums))
    
    S = D12 @ W @ D12

    # F*
    I = torch.eye(X.shape[0], device=device, dtype=X.dtype)
    temp = (I - alpha * S).float()
    F_star = torch.linalg.solve(temp, Y) # solve MF = Y for F, avoid direct inverse

    # label each point x_i as y_i = argamx F*_ij
    L = torch.argmax(F_star, dim=1)

    return L


In [38]:
def prepare_labels_label_propagation(len_X_support, len_X_query, y_support, device, num_classes=2):
    y_support = torch.tensor(y_support).long()
    zeros = torch.zeros((len_X_query, num_classes), dtype=torch.float, device=device)
    one_hot_y= nn.functional.one_hot(y_support, num_classes,).float().to(device)
    Y = torch.concatenate((one_hot_y, zeros))
    return Y

In [39]:
def scatterplot_propagated_labels(X, y1, y2):
    pca = PCA(n_components=2, random_state=0)
    X2 = pca.fit_transform(X)

    # Prepare subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

    for ax, (y, title) in zip(axes, [(y1, 'Labels → y1'), (y2, 'Labels → y2 (None→grey)')]):
        # Find unique real classes (ignore None/nan)
        classes = [c for c in np.unique(y) if c is not None and not (isinstance(c, float) and np.isnan(c))]
        # Plot each real class with default cycle
        for cls in classes:
            mask = (y == cls)
            ax.scatter(
                X2[mask, 0], X2[mask, 1],
                label=f'Class {cls}',
                s=20, alpha=0.5, edgecolors='face'
            )
        # Plot missing (None/nan) as grey
        missing_mask = [(v is None) or (isinstance(v, float) and np.isnan(v)) for v in y]
        if any(missing_mask):
            ax.scatter(
                X2[missing_mask, 0], X2[missing_mask, 1],
                label='Missing',
                color='lightgrey',
                s=20, alpha=0.1, edgecolors='face'
            )
        ax.set_title(title, fontsize=16)
        ax.set_xlabel('PC1', fontsize=14)
        ax.set_ylabel('PC2', fontsize=14)
        ax.legend(title='Label', fontsize=12, title_fontsize=12, loc='best')
        ax.grid(True)

    plt.tight_layout()
    plt.show()

In [40]:
# task_idx = 2
# support_loader, _, support_idx, query_idx = get_support_query_loaders(X_finetuning, y_ft1, task_idx, k=1, l=30, max_pos_query=10, seed=12576, batch_size=120, num_workers=0)
# print(y_ft1[support_idx, task_idx])
# Y = prepare_labels_label_propagation(len(support_idx), len(query_idx), y_ft1[support_idx, 2])
# episode_idx = np.concatenate((support_idx, query_idx))
# propagated_labels = label_propagation(torch.tensor(X_finetuning[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=0.0000000000000001)

In [41]:
# scatterplot_propagated_labels(X_finetuning[episode_idx], propagated_labels.cpu(), y_ft1[episode_idx, task_idx])

In [ ]:
def train_few_shot_model(model, X, y, task_idx, optimizer, device, writer, run, k, l, max_pos_query, seed, alpha, num_epochs=10, batch_size=120, num_workers=0):
    """
    model          : your NeuralNetwork instance
    X, y           : numpy arrays (N, D) and (N, T)
    criterion      : loss fn, e.g. BCEWithLogitsLoss()
    optimizer      : your torch optimizer
    device         : torch.device
    writer         : tensorboard writer
    run            : an identifier for the saved filename
    k, seed        : few-shot support size and RNG seed
    """
    model.to(device)
    best_score = 0.0
    file_path = f"./best_models/ft1_vanilla_{run}.pth"

    X = torch.from_numpy(X).float().to(device)
    y = torch.from_numpy(y).float().to(device)

    total_loss = 0
    count = 0

    for epoch in range(1, num_epochs + 1):
        model.train()

        # generate support and query sets
        support_loader, query_loader, support_idx, query_idx = get_support_query_loaders(X, y, task_idx, k, l, max_pos_query, seed+epoch, batch_size=batch_size, num_workers=num_workers)
        # ——— SUPPORT SET ———
        for inputs, labels in support_loader:
            # flatten if needed
            support_inputs = inputs.view(inputs.size(0), -1).to(device)
            support_labels = labels.to(device)

            optimizer.zero_grad()
            outputs_support = model(support_inputs)
            outputs_support = outputs_support.squeeze(1)

            loss_support = classification_loss(outputs_support, support_labels)


        # ——— QUERY SET ——— #
        all_labels_val = []
        all_preds_val = []

        for inputs, labels in query_loader:
            query_inputs = inputs.to(device)
            query_labels = labels.to(device)

            outputs_query = model(query_inputs)
            outputs_query = outputs_query.squeeze(1)

            loss_query = classification_loss(outputs_query, query_labels)

            all_labels_val.append(labels.cpu().numpy())
            all_preds_val.append(torch.sigmoid(outputs_query).detach().cpu().numpy())

        val_pr_auc, val_roc_auc, _, _ = getPrAucIndividualClassFinetuning(
            "Val", all_labels_val, all_preds_val, writer, epoch
        )

        # ——— LABEL PROPAGATION ——— # 
        episode_idx = np.concatenate((support_idx, query_idx))
        Y = prepare_labels_label_propagation(len(support_idx), len(query_idx), y[support_idx, task_idx], device)
        propagated_labels = label_propagation(torch.tensor(X[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=alpha)

        loss_label_prop = propagation_loss(propagated_labels[len(support_idx):].float(), query_labels.float())


        # ——— LOSS AND UPDATE STEP ——— # 
        loss_classification = (loss_query + loss_support) / (len(support_idx) + len(query_idx)) # mean Lc loss
        loss_combined = loss_label_prop + 0.5 * loss_classification
        optimizer.zero_grad()
        loss_combined.backward()
        optimizer.step()

        total_loss += loss_combined.item()
        count += 1

        avg_loss = total_loss/count
        print(f"[Epoch {epoch}/{num_epochs}]  Loss: {avg_loss:.4f}")

        # ——— SAVE BEST MODEL ——— # 
        saved = save_best_model(model, avg_pr_auc=val_pr_auc,best_score=best_score, file_path=file_path)
        if saved:
            best_score = val_pr_auc

        print(f"[Epoch {epoch}/{num_epochs}]  "
            f"Overall Loss: {avg_loss:.4f}, Query PR-AUC: {val_pr_auc:.4f}, Query ROC-AUC: {val_roc_auc:.4f}"
        )

    # ——— EVALUATE BEST MODEL ON QUERY SET ——— #
    model.load_state_dict(torch.load(file_path, map_location=device))
    model.to(device).eval()

    # generate support and query sets
    support_loader, query_loader, support_idx, query_idx = get_support_query_loaders(X, y, task_idx, k, l, max_pos_query, seed-4, batch_size=batch_size, num_workers=num_workers)

    all_labels_final = []
    all_preds_final = []

    with torch.no_grad():
        # ——— QUERY SET ——— #
        all_labels_val = []
        all_preds_val = []

        for inputs, labels in query_loader:
            query_inputs = inputs.to(device)
            query_labels = labels.to(device)

            outputs_query = model(query_inputs)
            outputs_query = outputs_query.squeeze(1)

            loss_query = classification_loss(outputs_query, query_labels)

            all_labels_final.append(labels.cpu().numpy())
            all_preds_final.append(torch.sigmoid(outputs_query).detach().cpu().numpy())

        best_pr_auc, best_roc_auc, pr_auc_per_task, roc_auc_per_task = getPrAucIndividualClassFinetuning(
            "Val", all_labels_final, all_preds_final, writer, epoch
        )

    print(
        f"=== Best Model Evaluation ===\n"
        f"Avg PR-AUC: {best_pr_auc:.4f}, Avg ROC-AUC: {best_roc_auc:.4f}"
    )

    return best_pr_auc, best_roc_auc, pr_auc_per_task, roc_auc_per_task


In [61]:
# hyperparameters & global vars
input_size = 2248 
hidden_sizes = [120, 48, None]
output_size = 9
batch_size = 120
lr = 0.001
k = 1
l = 10
num_epochs = 500
max_pos_query= 5
task_idx=2
alpha = 0.0000000000000001
dropout_rates = [0.3, 0.5] # input dropout, hidden dropout
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
seeds = [8479, 227, 5413, 8179, 7528]

# list to collect results avg over task
avg_results = {
        "Delta-AUC-PR": [],
        "ROC-AUC": []
    }

# list to collect results per task
all_results = {
        "Delta-AUC-PR": [],
        "ROC-AUC": []
    }  

# train and evaluate model 5 times
for run in range(5):
    # set seed
    seed_value = seeds[run]
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
    np.random.seed(seed_value)
    random.seed(seed_value)

    # initialize
    # writer = SummaryWriter(f"manual_runs/run_36_upsampling") # tensorboard
    writer = None
    model = NeuralNetwork(input_size, hidden_sizes, output_size, dropout_rates)

    # load best model
    state_dict = torch.load("./best_models/overall_best_vanilla.pth", map_location=device)
    model.load_state_dict(state_dict)

    # freeze all parameters except the last layer (fc3)
    for name, param in model.named_parameters():
        if not name.startswith("fc3"):
            param.requires_grad = False

    # check if all frozen:
    for name, param in model.named_parameters():
        print(name, param.requires_grad)
        
    # replace fc3 with a new Linear that has a single output neuron
    in_features = model.fc3.in_features # keep input features as they are
    model.fc3 = nn.Linear(in_features, 1) # replace output features
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr) # filter out all parameters except last layer

    # train model
    val_delta_auc_pr, val_roc_auc, val_pr_auc_per_task, val_roc_auc_per_task = train_few_shot_model(model, X_finetuning, y_ft1, task_idx, optimizer, device, writer, run, k, l, max_pos_query, seed_value, alpha, num_epochs=num_epochs, batch_size=120, num_workers=0)
    
    # collect results
    avg_results["Delta-AUC-PR"].append(val_delta_auc_pr)
    avg_results["ROC-AUC"].append(val_roc_auc)

    all_results["Delta-AUC-PR"].append(np.array(val_pr_auc_per_task))
    all_results["ROC-AUC"].append(np.array(val_roc_auc_per_task))
    
    # ------- for hyperparameter tuning ------- #
    # val_loss, val_delta_auc_pr, val_roc_auc = train_model(model, train_loader, val_loader, criterion, optimizer, device, writer, num_epochs=20)

    # hparams = {
    #     "lr": lr,
    #     "batch_size": batch_size,
    #     "optimizer": "Adam",
    #     "hidden_sizes_1": hidden_sizes[0],
    #     "hidden_sizes_2": hidden_sizes[1]
    # }
    # metrics = {
    #     "auc-pr": val_delta_auc_pr,
    #     "loss": val_loss
    # }
    # writer.add_hparams(hparams, metrics)
    # writer.close()


cuda
fc1.weight False
fc1.bias False
fc2.weight False
fc2.bias False
fc3.weight True
fc3.bias True
[2991 9763]
[Epoch 1/500]  Loss: 0.8704
Current avg PR AUC: -0.0949 did not improve over best score: 0.0000
[Epoch 1/500]  Overall Loss: 0.8704, Query PR-AUC: -0.0949, Query ROC-AUC: 0.4000
[13855  9785]
[Epoch 2/500]  Loss: 0.8898
Current avg PR AUC: -0.0217 did not improve over best score: 0.0000
[Epoch 2/500]  Overall Loss: 0.8898, Query PR-AUC: -0.0217, Query ROC-AUC: 0.5200
[13855  3498]


c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc\utils\training_finetuning1.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  support_dataset = TensorDataset(torch.tensor(Xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32))
c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc\utils\training_finetuning1.py:86: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  query_dataset = TensorDataset(torch.tensor(Xq, dtype=torch.float32),torch.tensor(yq, dtype=torch.float32))
C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\4066636454.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach()

[Epoch 3/500]  Loss: 0.9000
Current avg PR AUC: -0.1244 did not improve over best score: 0.0000
[Epoch 3/500]  Overall Loss: 0.9000, Query PR-AUC: -0.1244, Query ROC-AUC: 0.3200
[16571  9672]
[Epoch 4/500]  Loss: 0.8874
Current avg PR AUC: -0.1003 did not improve over best score: 0.0000
[Epoch 4/500]  Overall Loss: 0.8874, Query PR-AUC: -0.1003, Query ROC-AUC: 0.3600
[16571 12176]
[Epoch 5/500]  Loss: 0.8712
Saved new best model with avg PR AUC: 0.1060
[Epoch 5/500]  Overall Loss: 0.8712, Query PR-AUC: 0.1060, Query ROC-AUC: 0.5600
[13361 17501]
[Epoch 6/500]  Loss: 0.8598
Current avg PR AUC: 0.0917 did not improve over best score: 0.1060
[Epoch 6/500]  Overall Loss: 0.8598, Query PR-AUC: 0.0917, Query ROC-AUC: 0.5200
[1656 4465]
[Epoch 7/500]  Loss: 0.8554
Current avg PR AUC: 0.0818 did not improve over best score: 0.1060
[Epoch 7/500]  Overall Loss: 0.8554, Query PR-AUC: 0.0818, Query ROC-AUC: 0.5200
[1656 2419]
[Epoch 8/500]  Loss: 0.8470
Current avg PR AUC: -0.0440 did not improve 

C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\4066636454.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_support = torch.tensor(y_support).long()
C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\800776337.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  propagated_labels = label_propagation(torch.tensor(X[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=alpha)
c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc\utils\training_finetuning1.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTe

[Epoch 12/500]  Loss: 0.8383
Saved new best model with avg PR AUC: 0.3600
[Epoch 12/500]  Overall Loss: 0.8383, Query PR-AUC: 0.3600, Query ROC-AUC: 0.8800
[5297 9367]
[Epoch 13/500]  Loss: 0.8482
Current avg PR AUC: 0.0067 did not improve over best score: 0.3600
[Epoch 13/500]  Overall Loss: 0.8482, Query PR-AUC: 0.0067, Query ROC-AUC: 0.5600
[13855  2935]
[Epoch 14/500]  Loss: 0.8511
Current avg PR AUC: -0.1133 did not improve over best score: 0.3600
[Epoch 14/500]  Overall Loss: 0.8511, Query PR-AUC: -0.1133, Query ROC-AUC: 0.3600
[ 5297 17156]
[Epoch 15/500]  Loss: 0.8547
Current avg PR AUC: 0.0606 did not improve over best score: 0.3600
[Epoch 15/500]  Overall Loss: 0.8547, Query PR-AUC: 0.0606, Query ROC-AUC: 0.4400
[ 5297 10126]
[Epoch 16/500]  Loss: 0.8519
Current avg PR AUC: -0.1104 did not improve over best score: 0.3600
[Epoch 16/500]  Overall Loss: 0.8519, Query PR-AUC: -0.1104, Query ROC-AUC: 0.3600
[ 1656 11934]
[Epoch 17/500]  Loss: 0.8460
Current avg PR AUC: 0.1710 did 

...

C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\4066636454.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_support = torch.tensor(y_support).long()
C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\800776337.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  propagated_labels = label_propagation(torch.tensor(X[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=alpha)
c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc\utils\training_finetuning1.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTe

[Epoch 487/500]  Loss: 0.8135
Current avg PR AUC: -0.0806 did not improve over best score: 0.5000
[Epoch 487/500]  Overall Loss: 0.8135, Query PR-AUC: -0.0806, Query ROC-AUC: 0.4400
[ 2991 17338]
[Epoch 488/500]  Loss: 0.8134
Current avg PR AUC: 0.4633 did not improve over best score: 0.5000
[Epoch 488/500]  Overall Loss: 0.8134, Query PR-AUC: 0.4633, Query ROC-AUC: 0.9600
[ 8909 12293]
[Epoch 489/500]  Loss: 0.8136
Current avg PR AUC: 0.3322 did not improve over best score: 0.5000
[Epoch 489/500]  Overall Loss: 0.8136, Query PR-AUC: 0.3322, Query ROC-AUC: 0.7600
[13855   631]
[Epoch 490/500]  Loss: 0.8135
Current avg PR AUC: 0.2433 did not improve over best score: 0.5000
[Epoch 490/500]  Overall Loss: 0.8135, Query PR-AUC: 0.2433, Query ROC-AUC: 0.6400
[13361  2223]
[Epoch 491/500]  Loss: 0.8133
Current avg PR AUC: -0.1073 did not improve over best score: 0.5000
[Epoch 491/500]  Overall Loss: 0.8133, Query PR-AUC: -0.1073, Query ROC-AUC: 0.3600
[ 4600 12471]
[Epoch 492/500]  Loss: 0.8

C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\4066636454.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_support = torch.tensor(y_support).long()
C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\800776337.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  propagated_labels = label_propagation(torch.tensor(X[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=alpha)
c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc\utils\training_finetuning1.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTe

[Epoch 493/500]  Loss: 0.8131
Current avg PR AUC: 0.2368 did not improve over best score: 0.5000
[Epoch 493/500]  Overall Loss: 0.8131, Query PR-AUC: 0.2368, Query ROC-AUC: 0.6800
[16571  3958]
[Epoch 494/500]  Loss: 0.8130
Current avg PR AUC: -0.0356 did not improve over best score: 0.5000
[Epoch 494/500]  Overall Loss: 0.8130, Query PR-AUC: -0.0356, Query ROC-AUC: 0.5200
[ 2991 12847]
[Epoch 495/500]  Loss: 0.8132
Current avg PR AUC: -0.0698 did not improve over best score: 0.5000
[Epoch 495/500]  Overall Loss: 0.8132, Query PR-AUC: -0.0698, Query ROC-AUC: 0.4000
[16571 10853]
[Epoch 496/500]  Loss: 0.8129
Current avg PR AUC: -0.0921 did not improve over best score: 0.5000
[Epoch 496/500]  Overall Loss: 0.8129, Query PR-AUC: -0.0921, Query ROC-AUC: 0.4000
[2991 4005]
[Epoch 497/500]  Loss: 0.8130
Current avg PR AUC: 0.2227 did not improve over best score: 0.5000
[Epoch 497/500]  Overall Loss: 0.8130, Query PR-AUC: 0.2227, Query ROC-AUC: 0.6400
[2991 4973]
[Epoch 498/500]  Loss: 0.813

C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\4066636454.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_support = torch.tensor(y_support).long()
C:\Users\sonja\AppData\Local\Temp\ipykernel_23340\800776337.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  propagated_labels = label_propagation(torch.tensor(X[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=alpha)
c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc\utils\training_finetuning1.py:85: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTe

In [62]:
filepath_normal = "./metrics/vanilla_ft1.json"
filepath_per_task = "./metrics/vanilla_ft1_per_task.json"
filepath_per_run = "./metrics/vanilla_ft1_per_run.json"
log_results_nn(avg_results, all_results, filepath_normal, filepath_per_run, filepath_per_task)